In [6]:
# the purpose of this notebook is to make inference for a formula

In [1]:
# import data

from pathlib import Path
import pandas as pd

# Load the compiled data
project_root = Path.cwd().resolve().parent if 'notebooks' in str(Path.cwd()) else Path.cwd().resolve()
excel_path = project_root / 'data' / 'raw' / 'Compiled Data.xlsx'

df = pd.read_excel(excel_path)


# Show the 25 original formulas and their three measured targets from df.
formula_cols = ["TEM", "Joncryl", "NC513", "PEG-DE", "NGDE"]
measured_target_cols = [
    "Avg_Density_g_cm3",
    "Dry Strength Mean",
    "Wet Strength Mean",
]
df = df[formula_cols + measured_target_cols].copy()
df[formula_cols] = df[formula_cols].fillna(0.0)
df[measured_target_cols] = df[measured_target_cols].apply(
    pd.to_numeric, errors="raise"
)

assert len(df) == 25, (
    f"Expected the 25 original samples; found {len(df)}."
)
display(df)

,TEM,Joncryl,NC513,PEG-DE,NGDE,Avg_Density_g_cm3,Dry Strength Mean,Wet Strength Mean
0,8,0.0,0.0,0.0,0.0,0.192000,53.674733,23.680000
1,8,0.7,0.0,0.0,0.0,0.195000,68.863717,24.258481
2,8,1.0,0.0,0.0,0.0,0.159000,50.405845,21.329450
3,8,1.5,0.0,0.0,0.0,0.159933,54.717605,20.183592
4,8,2.0,0.0,0.0,0.0,0.201000,85.787785,27.585357
5,8,3.0,0.0,0.0,0.0,0.202000,80.535443,27.921901
6,5,0.0,0.0,0.0,0.0,0.298930,157.659876,51.133703
7,5,0.0,0.7,0.0,0.0,0.360035,215.989325,82.812461
8,5,0.0,1.0,0.0,0.0,0.355557,218.543779,77.134354
9,5,0.0,2.0,0.0,0.0,0.358055,196.341194,76.406460


In [2]:
# load the gpr_models
import joblib
from pathlib import Path

project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
models_dir = project_root / "models"
gpr_models = joblib.load(models_dir / "gpr_models.joblib")

In [5]:
# make inference for a formula from he gpr_models

import numpy as np
import pandas as pd

feature_cols = ["TEM", "Joncryl", "NC513", "PEG-DE", "NGDE"]

# Enter the formula amounts here.
formula_values = {
    "TEM": 8,
    "Joncryl": 1,
    "NC513": 0.0,
    "PEG-DE": 0,
    "NGDE": 0.0,
}

missing_features = [col for col in feature_cols if col not in formula_values]
if missing_features:
    raise ValueError(f"Enter values for these formula features: {missing_features}")

X_formula = pd.DataFrame(
    [[formula_values[col] for col in feature_cols]],
    columns=feature_cols,
)

if not np.isfinite(X_formula.to_numpy(dtype=float)).all():
    raise ValueError("Formula values must all be finite numbers.")

results = []

for name, model in gpr_models.items():
    mean, std = model.predict(X_formula, return_std=True)
    results.append({
        "Property": name,
        "Prediction": float(np.asarray(mean).reshape(-1)[0]),
        "Predictive std": float(np.asarray(std).reshape(-1)[0]),
    })

display(pd.DataFrame(results))

,Property,Prediction,Predictive std
0,Avg_Density_g_cm3,0.159836,0.011639
1,Dry Strength Mean,53.914618,11.541908
2,Wet Strength Mean,23.671018,3.805434
